# Category Classification — Training Pipeline

**Responsibility:** retrieve features from the Hopsworks Feature Store → fit preprocessors → train and evaluate models → save `category_best_model.pkl`.

This notebook implements the **Training** stage of the FTI (Feature-Training-Inference) pipeline. It consumes engineered features produced by the Feature Pipeline (`category_classification_f.ipynb`), trains multiple model architectures (Tier-1 structured-only and Tier-2 structured + text), compares them, and persists the best artifact for the Inference Pipeline (`category_classification_i.ipynb`).

| Input | Description |
|---|---|
| Hopsworks Feature Group `category_features` (v1) | Engineered features with split labels |
| `category_metadata.json` | Label maps and feature-column lists produced during feature engineering |

| Output | Description |
|---|---|
| `category_best_model.pkl` | Serialised pipeline + preprocessors + metadata + label maps |
| Hopsworks Model Registry `category_classifier` | Versioned model artifact with metrics |

> **Prerequisites:** Feature Pipeline (`category_classification_f.ipynb`) must have run to populate the Feature Store and metadata file.
> **Next stage:** run `category_classification_i.ipynb` (Inference Pipeline) which loads the saved artifact and scores new batches.

## Shared Imports & Configuration

Import standard-library and third-party packages used across the entire Training Pipeline. This cell also configures global plotting defaults, suppresses warnings for cleaner output, and sets pandas display options so wide DataFrames render correctly in the notebook.

In [ ]:
import json
import os
import pickle
import random
import shutil
import warnings
from pathlib import Path

import numpy as np               # Numerical arrays and random-seed control
import pandas as pd              # DataFrames for feature manipulation
import matplotlib.pyplot as plt  # Plotting backend
import seaborn as sns            # Statistical visualisations (heatmaps, bar plots)
import scipy.sparse as sp        # Sparse-matrix checks for TF-IDF output

from IPython.display import display          # Pretty-print DataFrames in notebook cells
from sklearn.compose import ColumnTransformer  # Multi-column preprocessing pipelines
from sklearn.dummy import DummyClassifier      # Baseline model (most-frequent class)
from sklearn.experimental import enable_halving_search_cv  # noqa — enables HalvingRandomSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer  # Text → sparse TF-IDF vectors
from sklearn.linear_model import LogisticRegression          # Linear baseline classifier
from sklearn.metrics import (
    balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import (
    HalvingRandomSearchCV, StratifiedKFold,  # Hyper-parameter search + cross-validation folds
    cross_val_score,                         # Quick CV scoring helper
)
from sklearn.pipeline import Pipeline                     # Chain preprocessing + classifier
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, TargetEncoder

from lightgbm import LGBMClassifier  # Gradient-boosted trees (primary model family)

import hopsworks  # Hopsworks Feature Store and Model Registry client
import tomllib  # Python 3.11+ std-lib TOML parser
from category_classification import (
    make_dist, tfidf_kw,
    TwoStageClassifier, evaluate,
    predict_dispatch,
    fp3_platform_worker,
)
import gc
from matplotlib.patches import Patch
from IPython.display import display, Markdown
from datetime import datetime


# Suppress noisy convergence / deprecation warnings during grid search
warnings.filterwarnings("ignore")
# Global figure defaults for all matplotlib plots generated in this notebook
plt.rcParams.update({"figure.figsize": (14, 6), "figure.dpi": 100, "font.size": 11})
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)
print("Imports OK.")

## TOML Configuration & Distribution Helper

Load the central `category_classification_fti.toml` configuration file. This file defines paths, Hopsworks connection details, feature-engineering parameters, model hyper-parameter search spaces, and reproducibility settings. We also import reusable helpers (`make_dist`, `tfidf_kw`, `TwoStageClassifier`, `evaluate`, `predict_dispatch`, `fp3_platform_worker`) from the shared module `category_classification.py` so they stay in version control rather than inline notebook code.

In [ ]:
# Import project-specific helpers from the shared Python module
# (imported in the first cell above)

CFG_PATH = Path("category_classification_fti.toml")
with open(CFG_PATH, "rb") as f:
    cfg = tomllib.load(f)  # Load the full configuration tree into a dict

# LightGBM can run on CPU or GPU depending on environment / config
_LGB_DEVICE = cfg["models"]["lightgbm"].get("device", "cpu")
print(f"Config loaded from {CFG_PATH}")
print(f"LightGBM device: {_LGB_DEVICE}")

## Path, Constant & Service Definitions

Resolve every file-system path, reproducibility seed, domain constant (e.g. the name of the "OTHER" catch-all category), and Hopsworks Feature Group / Feature View identifier from the loaded TOML config. Centralising these values means the rest of the notebook is declarative: change the TOML and the entire pipeline adapts without touching code.

**Outputs:** `MODEL_ARTIFACT`, `METADATA_PATH`, `RANDOM_STATE`, `HOPSWORKS_HOST`, `FG_NAME`, `FV_NAME`, `MODEL_NAME`, and many more.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Resolve output paths from the TOML [paths] section
_paths = cfg["paths"]
MODEL_ARTIFACT    = Path(_paths["model_artifact"])    # e.g. category_best_model.pkl
MODEL_EXPORT_DIR  = Path(_paths["model_export_dir"])  # Temporary staging for Hopsworks upload
CV_RESULTS_PATH   = Path(_paths["cv_results"])        # Optional CV results cache
PREDICTIONS_PATH  = Path(_paths["predictions"])       # Optional inference-output cache
METADATA_PATH     = Path(_paths["metadata"])          # JSON with label maps & column lists

# ── Reproducibility ────────────────────────────────────────────────────────────
_repr = cfg["reproducibility"]
RANDOM_STATE  = _repr["random_state"]   # Fixed seed for every stochastic operation
N_CV_FOLDS    = _repr["n_cv_folds"]     # Number of stratified CV folds
# Seed NumPy and Python std-lib random so splits, sampling, and LightGBM are deterministic
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ── Domain constants ───────────────────────────────────────────────────────────
_domain = cfg["domain"]
OTHER_CAT  = _domain["other_category"]  # Human-readable name of the catch-all class
PREFIX     = _domain["prefix"]          # Prefix used for feature naming
TEXT_COL   = _domain["text_col"]        # Primary free-text column
TEXT_COL_2 = _domain["text_col_2"]      # Secondary free-text column
TEXT_COL_3 = _domain["text_col_3"]      # Tertiary free-text column
# High-cardinality columns are target-encoded; low-cardinality ones are ordinal-encoded
HIGH_CARD  = cfg["features"]["categorical"]["encoding"]["high_cardinality"]

# ── Hopsworks ─────────────────────────────────────────────────────────────────
# Read connection params from env vars (preferred for CI/CD) or fall back to TOML
_hw = cfg["hopsworks"]
HOPSWORKS_HOST    = os.environ.get("HOPSWORKS_HOST",    _hw["host"])
HOPSWORKS_PROJECT = os.environ.get("HOPSWORKS_PROJECT", _hw["project"])
HOPSWORKS_API_KEY = os.environ.get("HOPSWORKS_API_KEY", _hw["api_key"])
FG_NAME           = _hw["feature_group"]["name"]
FG_VERSION        = _hw["feature_group"]["version"]
# Text feature groups store large text fields separately to avoid bloating the struct FG
FG_TEXT_1_NAME    = _hw.get("feature_group_text_1", {}).get("name",    FG_NAME + "_text_1")
FG_TEXT_1_VERSION = _hw.get("feature_group_text_1", {}).get("version", 1)
FG_TEXT_2_NAME    = _hw.get("feature_group_text_2", {}).get("name",    FG_NAME + "_text_2")
FG_TEXT_2_VERSION = _hw.get("feature_group_text_2", {}).get("version", 1)
FG_TEXT_3_NAME    = _hw.get("feature_group_text_3", {}).get("name",    FG_NAME + "_text_3")
FG_TEXT_3_VERSION = _hw.get("feature_group_text_3", {}).get("version", 1)
FV_NAME           = _hw.get("feature_view", {}).get("name",    FG_NAME + "_view")
FV_VERSION        = _hw.get("feature_view", {}).get("version", 1)
MODEL_NAME        = _hw["model_registry"]["name"]


print(f"Metadata  : {METADATA_PATH}")
print(f"Model     : {MODEL_ARTIFACT}")
print(f"Hopsworks : {HOPSWORKS_HOST}  project={HOPSWORKS_PROJECT}")
print(f"  FG (struct) : {FG_NAME} v{FG_VERSION}")
print(f"  FG (text 1) : {FG_TEXT_1_NAME} v{FG_TEXT_1_VERSION}")
print(f"  FG (text 2) : {FG_TEXT_2_NAME} v{FG_TEXT_2_VERSION}")
print(f"  FG (text 3) : {FG_TEXT_3_NAME} v{FG_TEXT_3_VERSION}")
print(f"  FV          : {FV_NAME} v{FV_VERSION}")
print(f"  Model       : {MODEL_NAME}")

---
# Part 2 — Training Pipeline

**Responsibility:** retrieve features from the Hopsworks Feature Store → fit preprocessors
→ train and evaluate models → save `category_best_model.pkl`.

This is the main training body. You can re-run from this point after tweaking model
architecture, hyper-parameters, or evaluation logic without re-executing the Feature
Pipeline (provided the Feature Store and metadata file are still intact).

**Inputs:**
- Hopsworks Feature Group `category_features` (v1) and the three text satellite FGs
- `category_metadata.json` (label maps, column lists, split configuration)

**Outputs:**
- `category_best_model.pkl` — serialised scikit-learn Pipeline + preprocessors + metadata
- Hopsworks Model Registry entry — versioned artifact with attached metrics

## TP-1 · Load Features from Hopsworks Feature Store

Connect to Hopsworks, materialise the Feature View, and build chronological train/test
splits that mirror the split logic used in the Feature Pipeline (FP-3). This cell also
recomputes platform-level rolling statistics **on the training split only** to prevent
data leakage, drops classes with too few samples for stratified CV, and constructs:

- `X_tr` / `X_te` — Tier-1 structured-feature matrices
- `X_tr_t2` / `X_te_t2` — Tier-2 matrices (structured + the three text columns)
- `y_tr` / `y_te` — integer-encoded category labels

In [ ]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"{METADATA_PATH} not found — run _f notebook first.")

# Load metadata produced by the Feature Pipeline (label maps, column lists, FG versions)
with open(METADATA_PATH) as f:
    meta = json.load(f)

# Unpack metadata into notebook-level variables so downstream cells stay clean
target_labels        = meta["target_labels"]
label_map            = meta["label_map"]
label_map_inv        = {int(k): v for k, v in meta["label_map_inv"].items()}
other_idx            = meta["other_idx"]
n_classes            = meta["n_classes"]
short_other          = meta["short_other"]
FEATURE_COLS         = meta["FEATURE_COLS"]
CATEGORICAL_FEATURES = meta["CATEGORICAL_FEATURES"]
NUMERICAL_FEATURES   = meta["NUMERICAL_FEATURES"]
HIGH_CARD            = meta["HIGH_CARD"]
LOW_CARD             = meta["LOW_CARD"]
TEXT_COL             = meta["TEXT_COL"]
TEXT_COL_2           = meta.get("TEXT_COL_2", "incompatible_content_explanation")
TEXT_COL_3           = meta.get("TEXT_COL_3", "decision_facts")
# Keep FG/FV names from metadata, falling back to TOML-derived constants if absent
_fg_name             = meta["fg_name"]
_fg_version          = meta["fg_version"]
_fg_text_1_name      = meta.get("fg_text_1_name",    FG_TEXT_1_NAME)
_fg_text_1_version   = meta.get("fg_text_1_version", FG_TEXT_1_VERSION)
_fg_text_2_name      = meta.get("fg_text_2_name",    FG_TEXT_2_NAME)
_fg_text_2_version   = meta.get("fg_text_2_version", FG_TEXT_2_VERSION)
_fg_text_3_name      = meta.get("fg_text_3_name",    FG_TEXT_3_NAME)
_fg_text_3_version   = meta.get("fg_text_3_version", FG_TEXT_3_VERSION)
_fv_name             = meta.get("fv_name",    FV_NAME)
_fv_version          = meta.get("fv_version", FV_VERSION)
_fp                  = cfg["feature_pipeline"]

# ── Functions for extraction steps ────────────────────────────────────────────

def _ensure_target_enc(df_all, fg_struct):
    """Fetch target_enc from the struct FG when Hopsworks batch data omits it.

    Mutates *df_all* by left-merging on ``row_id``.
    """
    print("  target_enc missing from batch data — fetching from struct FG...")
    _target_df = fg_struct.select(["row_id", "target_enc"]).read(dataframe_type="pandas")
    return df_all.merge(_target_df, on="row_id", how="left")


def _drop_unparseable_dates(df_all, _app_ts):
    """Drop rows whose application_date could not be parsed.

    Returns the filtered ``(df_all, _app_ts)``.
    """
    _valid_mask = _app_ts.notna()
    _n_dropped = (~_valid_mask).sum()
    if _n_dropped:
        print(f"  Dropping {_n_dropped} row(s) with unparseable application_date.")
        df_all = df_all[_valid_mask].copy()
        _app_ts = _app_ts[_valid_mask]
    return df_all, _app_ts


def _drop_sparse_classes(X_tr_all, X_te_all, y_tr, y_te,
                         target_labels, label_map, label_map_inv,
                         n_classes, short_other, other_idx):
    """Remove classes with fewer than 2 training samples and remap labels.

    Returns the updated ``(X_tr_all, X_te_all, y_tr, y_te, target_labels,
    label_map, label_map_inv, n_classes, other_idx)``.
    """
    _tr_vc = pd.Series(y_tr).value_counts()
    # Any class with < 2 train samples OR present only in test cannot be stratified
    _too_few = set(_tr_vc[_tr_vc < 2].index) | (set(np.unique(y_te)) - set(_tr_vc.index))
    if not _too_few:
        # Nothing to drop — return the original objects unchanged.
        return (X_tr_all, X_te_all, y_tr, y_te,
                target_labels, label_map, label_map_inv, n_classes, other_idx)
    _drop_names = [label_map_inv[int(c)] for c in sorted(_too_few)]
    print(f"Dropping {len(_too_few)} class(es) with < 2 training samples: {_drop_names}")
    _keep_tr = ~np.isin(y_tr, list(_too_few))
    _keep_te = ~np.isin(y_te, list(_too_few))
    X_tr_all = X_tr_all[_keep_tr].reset_index(drop=True)
    X_te_all = X_te_all[_keep_te].reset_index(drop=True)
    y_tr = y_tr[_keep_tr]
    y_te = y_te[_keep_te]
    # Build a contiguous 0..N-1 mapping so classifiers receive dense labels
    _old_to_new = {old: new for new, old in enumerate(sorted(set(y_tr) | set(y_te)))}
    y_tr = np.array([_old_to_new[v] for v in y_tr], dtype=int)
    y_te = np.array([_old_to_new[v] for v in y_te], dtype=int)
    target_labels = [label_map_inv[int(old)] for old in sorted(_old_to_new.keys())]
    label_map = {lbl: i for i, lbl in enumerate(target_labels)}
    label_map_inv = {i: lbl for lbl, i in label_map.items()}
    n_classes = len(target_labels)
    other_idx = label_map[short_other]
    print(f"  Remapped to {n_classes} classes. OTHER index = {other_idx}")
    return (X_tr_all, X_te_all, y_tr, y_te,
            target_labels, label_map, label_map_inv, n_classes, other_idx)


def _recompute_platform_features(X_tr_all, X_te_all, X_tr, X_te, _fp):
    """Recompute platform rolling features and automation rate on the train split.

    Avoids leakage from the Feature Store's original chronological split.
    Mutates *X_tr* and *X_te* in-place (adds ``platform_volume_3h``,
    ``platform_auto_fraction_3h``, ``platform_content_type_div_3h``,
    ``platform_automation_rate``).

    Returns the ``PLATFORM_AUTOMATION_RATES`` dict for the model artifact.
    """
    PLATFORM_AUTOMATION_RATES = {}
    _ts_col = "created_at" if "created_at" in X_tr_all.columns else "application_date"
    if _ts_col not in X_tr_all.columns:
        print("  WARNING: no timestamp column found — platform rolling features left as-is.")
        return PLATFORM_AUTOMATION_RATES

    # Build auxiliary DataFrames with the platform, timestamp, and automation flag
    _tr_aux = pd.DataFrame({
        "platform_name": X_tr_all["platform_name"].values,
        "_ts": pd.to_datetime(X_tr_all[_ts_col].values, errors="coerce"),
        "_auto": (X_tr_all["automated_detection"].fillna("").str.lower() == "yes").astype(np.float32).values,
        "_ctp": X_tr_all["content_type_primary"].fillna("UNKNOWN").values,
        "_istr": True,
        "_pos": np.arange(len(X_tr_all), dtype=np.int64),
    }, index=X_tr_all.index)
    _te_aux = pd.DataFrame({
        "platform_name": X_te_all["platform_name"].values,
        "_ts": pd.to_datetime(X_te_all[_ts_col].values, errors="coerce"),
        "_auto": (X_te_all["automated_detection"].fillna("").str.lower() == "yes").astype(np.float32).values,
        "_ctp": X_te_all["content_type_primary"].fillna("UNKNOWN").values,
        "_istr": False,
        "_pos": np.arange(len(X_te_all), dtype=np.int64) + len(X_tr_all),
    }, index=X_te_all.index)
    _combo_aux = pd.concat([_tr_aux, _te_aux])

    _valid_mask = _combo_aux["_ts"].notna().values
    _n_nat = (~_valid_mask).sum()
    if _n_nat:
        print("  INFO: {} row(s) with unparseable timestamp — using defaults for platform rolling features.".format(_n_nat))

    # Rolling window length in nanoseconds for vectorised time-delta comparisons
    _WIN_NS = np.int64(_fp.get("rolling_window_hours", 3) * 3600 * 1_000_000_000)
    _max_pos = int(_combo_aux["_pos"].max()) + 1
    _vol_buf = np.zeros(_max_pos, dtype=np.float32)
    _af_buf  = np.zeros(_max_pos, dtype=np.float32)
    _div_buf = np.ones(_max_pos, dtype=np.float32)

    # Compute rolling aggregates per platform using the same worker as the Feature Pipeline
    for _plat, _grp in _combo_aux[_valid_mask].groupby("platform_name", sort=False):
        _r = fp3_platform_worker((_plat, _grp), _WIN_NS, "all_prior")
        _tr_p, _vol_tr, _af_tr, _div_tr, _te_p, _vol_te, _af_te, _div_te = _r
        if len(_tr_p):
            _vol_buf[_tr_p] = _vol_tr
            _af_buf[_tr_p]  = _af_tr
            _div_buf[_tr_p] = _div_tr
        if len(_te_p):
            _vol_buf[_te_p] = _vol_te
            _af_buf[_te_p]  = _af_te
            _div_buf[_te_p] = _div_te

    # Write the recomputed buffers back into the Tier-1 DataFrames
    _tr_pos = _tr_aux["_pos"].values
    _te_pos = _te_aux["_pos"].values
    X_tr["platform_volume_3h"]           = _vol_buf[_tr_pos]
    X_tr["platform_auto_fraction_3h"]    = _af_buf[_tr_pos]
    X_tr["platform_content_type_div_3h"] = _div_buf[_tr_pos]
    X_te["platform_volume_3h"]           = _vol_buf[_te_pos]
    X_te["platform_auto_fraction_3h"]    = _af_buf[_te_pos]
    X_te["platform_content_type_div_3h"] = _div_buf[_te_pos]

    # Derive per-platform automation rate from the training split only (no leakage)
    _train_rate = _tr_aux["_auto"].groupby(_tr_aux["platform_name"]).mean()
    PLATFORM_AUTOMATION_RATES = _train_rate.to_dict()
    X_tr["platform_automation_rate"] = X_tr["platform_name"].map(_train_rate).fillna(np.float32(0)).values
    X_te["platform_automation_rate"] = X_te["platform_name"].map(_train_rate).fillna(np.float32(0)).values

    print("  Recomputed platform_volume_3h, platform_auto_fraction_3h, "
          "platform_content_type_div_3h, platform_automation_rate "
          "on training split (automation_rate mean={:.3f})".format(X_tr['platform_automation_rate'].mean()))
    del _tr_aux, _te_aux, _combo_aux, _vol_buf, _af_buf, _div_buf, _train_rate
    return PLATFORM_AUTOMATION_RATES


# ── Connect to Hopsworks ──────────────────────────────────────────────────────
project = hopsworks.login(
    host=HOPSWORKS_HOST,
    project=HOPSWORKS_PROJECT,
    api_key_value=HOPSWORKS_API_KEY,
)
fs = project.get_feature_store()
print(f"Connected  : {HOPSWORKS_HOST}  →  project '{project.name}'")

# ── Get Feature View (recreate if deleted) ────────────────────────────────────
# The Feature View joins the struct FG with the three text FGs on row_id.
try:
    fv = fs.get_feature_view(name=_fv_name, version=_fv_version)
except Exception:
    fv = None
if fv is None:
    fg_struct = fs.get_feature_group(name=_fg_name, version=_fg_version)
    fg_text_1 = fs.get_feature_group(name=_fg_text_1_name, version=_fg_text_1_version)
    fg_text_2 = fs.get_feature_group(name=_fg_text_2_name, version=_fg_text_2_version)
    fg_text_3 = fs.get_feature_group(name=_fg_text_3_name, version=_fg_text_3_version)
    try:
        _fv_query = (
            fg_struct.select_all()
            .join(fg_text_1.select([TEXT_COL]),   on=["row_id"], join_type="left")
            .join(fg_text_2.select([TEXT_COL_2]), on=["row_id"], join_type="left")
            .join(fg_text_3.select([TEXT_COL_3]), on=["row_id"], join_type="left")
        )
    except AttributeError:
        # Fallback for older Hopsworks SDK versions without select_all()
        _struct_cols = [feat.name for feat in fg_struct.features]
        _fv_query = (
            fg_struct.select(_struct_cols)
            .join(fg_text_1.select([TEXT_COL]),   on=["row_id"], join_type="left")
            .join(fg_text_2.select([TEXT_COL_2]), on=["row_id"], join_type="left")
            .join(fg_text_3.select([TEXT_COL_3]), on=["row_id"], join_type="left")
        )
    fv = fs.get_or_create_feature_view(
        name=_fv_name,
        version=_fv_version,
        query=_fv_query,
        description="Category features view for chronological train/test splitting",
    )
    print(f"Feature View '{_fv_name}' v{_fv_version} recreated.")
else:
    print(f"Feature View '{_fv_name}' v{_fv_version} loaded.")
print(f"  Struct FG : '{_fg_name}' v{_fg_version}")
print(f"  Text FG 1 : '{_fg_text_1_name}' v{_fg_text_1_version}")
print(f"  Text FG 2 : '{_fg_text_2_name}' v{_fg_text_2_version}")
print(f"  Text FG 3 : '{_fg_text_3_name}' v{_fg_text_3_version}")

# ── Chronological train/test split (mirrors FP-3) ────────────────────────────
print("Reading all data from Feature View for chronological split...")
df_all = fv.get_batch_data(dataframe_type="pandas")

# Guard: if the target column was dropped by Hopsworks, back-fill it from the struct FG
if "target_enc" not in df_all.columns:
    df_all = _ensure_target_enc(df_all, fg_struct)

# Parse application_date and drop rows that cannot be ordered chronologically
_app_ts = pd.to_datetime(df_all["application_date"], errors="coerce")
df_all, _app_ts = _drop_unparseable_dates(df_all, _app_ts)
_sorted_pos = _app_ts.argsort().values
_n_tr = int(len(_sorted_pos) * (1 - _fp["test_size"]))
_tr_pos = _sorted_pos[:_n_tr]
_te_pos = _sorted_pos[_n_tr:]
X_tr_all = df_all.iloc[_tr_pos].reset_index(drop=True)
X_te_all = df_all.iloc[_te_pos].reset_index(drop=True)
print(f"Chronological split:  Train {len(X_tr_all):,}  Test {len(X_te_all):,}")

# Extract integer-encoded targets before dropping non-feature columns
y_tr = X_tr_all["target_enc"].values.astype(int)
y_te = X_te_all["target_enc"].values.astype(int)

# row_id is only needed for joins; remove it from model inputs
if "row_id" in X_tr_all.columns:
    X_tr_all = X_tr_all.drop(columns=["row_id"])
    X_te_all = X_te_all.drop(columns=["row_id"])

# Drop classes with < 2 training samples and re-index labels contiguously
(X_tr_all, X_te_all, y_tr, y_te,
 target_labels, label_map, label_map_inv,
 n_classes, other_idx) = _drop_sparse_classes(
     X_tr_all, X_te_all, y_tr, y_te,
     target_labels, label_map, label_map_inv,
     n_classes, short_other, other_idx,
 )

# Structured-feature DataFrames for Tier-1 models
_missing_num = _fp["missing_numeric"]
X_tr = pd.DataFrame(index=X_tr_all.index)
X_te = pd.DataFrame(index=X_te_all.index)
for c in FEATURE_COLS:
    X_tr[c] = X_tr_all[c].values if c in X_tr_all.columns else _missing_num
    X_te[c] = X_te_all[c].values if c in X_te_all.columns else _missing_num

# ── Recompute platform features on the train split (no leakage) ───────────────
PLATFORM_AUTOMATION_RATES = {}
if all(c in X_tr_all.columns for c in ["platform_name", "automated_detection", "content_type_primary"]):
    PLATFORM_AUTOMATION_RATES = _recompute_platform_features(
        X_tr_all, X_te_all, X_tr, X_te, _fp,
    )
else:
    print("  WARNING: missing platform/aux columns — platform features left as-is.")

# T2 views: structured features + all three text columns side-by-side.
X_tr_t2 = X_tr.copy()
X_te_t2 = X_te.copy()
for col in (TEXT_COL, TEXT_COL_2, TEXT_COL_3):
    X_tr_t2[col] = (X_tr_all[col].fillna("") if col in X_tr_all.columns
                    else pd.Series("", index=X_tr_all.index)).values
    X_te_t2[col] = (X_te_all[col].fillna("") if col in X_te_all.columns
                    else pd.Series("", index=X_te_all.index)).values

# Free the full-feature DataFrames to reduce memory before model training
del X_tr_all, X_te_all, df_all
gc.collect()

print(f"Features loaded — FV '{_fv_name}' v{_fv_version}")
print(f"  Train : {len(X_tr):,}   Test : {len(X_te):,}   Classes : {n_classes}")
print(f"  Structured features ({len(FEATURE_COLS)}): {FEATURE_COLS[:5]} ...")
print(f"  Text columns: {TEXT_COL!r}  {TEXT_COL_2!r}  {TEXT_COL_3!r}")



## TP-2 · Fit Preprocessors on Training Data

Preprocessors are training artifacts — they must be fit exclusively on training rows
to prevent data leakage. We build two ColumnTransformers:

1. **Tier-1 (`preprocessor_t1`)** — ordinal encoding for low-cardinality categoricals,
   target encoding for high-cardinality categoricals, passthrough for numericals.
2. **Tier-2 (`preprocessor_t2`)** — same as T1 plus three TF-IDF vectorisers for the
   free-text columns.

Both are stored in `category_best_model.pkl` so the Inference Pipeline can transform
raw rows without any connection to the Feature Store.

In [ ]:
# ── Tier-1 preprocessor: structured features only ─────────────────────────────
# Read encoder parameters from the TOML config tree
_enc = cfg["features"]["categorical"]["encoding"]
_ord_kw = _enc["ordinal_encoder_params"]
_tgt_kw = _enc["target_encoder_params"]

# ColumnTransformer routes each column list to its designated transformer
preprocessor_t1 = ColumnTransformer([
    ("ord",  OrdinalEncoder(**_ord_kw), LOW_CARD),
    ("tgt",  TargetEncoder(**_tgt_kw),  HIGH_CARD),
    ("num",  "passthrough",             NUMERICAL_FEATURES),
], remainder="drop")
preprocessor_t1.fit(X_tr, y_tr)
print("Tier-1 preprocessor fitted.")

# ── Tier-2 preprocessor: structured + TF-IDF on all three text columns ─────────
_txt = cfg["features"]["text"]
# Issue 11 fix: validate text column fill rates before fitting TF-IDF.
# All-empty text produces a zero-sparse matrix, making Tier-2 a Tier-1 with extra cost.
_txt_min_fill = cfg["features"]["text"].get("min_fill_rate", 0.05)
for _tcol in (TEXT_COL, TEXT_COL_2, TEXT_COL_3):
    _fill = (X_tr_t2[_tcol].fillna("") != "").mean()
    print(f"  {_tcol}: {_fill:.1%} fill rate (train)")
    if _fill < _txt_min_fill:
        raise ValueError(
            f"Text column '{_tcol}' has fill rate {_fill:.1%} < {_txt_min_fill:.0%} in "
            "training data. TF-IDF would train on near-zero signal. "
            "Check that text columns were populated correctly in FP-1."
        )

# tfidf_kw imported from category_classification builds the kwargs dict per column
# ColumnTransformer.fit() internally calls fit_transform(), accumulating all
# sub-transformer outputs simultaneously (~2.8 GB Xs + ~2.8 GB hstack on 2.8 M rows).
# Fix: structural fit on a tiny sample sets sklearn metadata; then each sub-transformer
# is fitted individually (one peak at a time); fitted estimators are injected into
# transformers_ so downstream transform() and Pipeline CV calls work unchanged.
_SAMPLE = cfg["feature_pipeline"].get("preprocessor_fit_sample", 1_000)
preprocessor_t2 = ColumnTransformer([
    ("ord",       OrdinalEncoder(**_ord_kw), LOW_CARD),
    ("tgt",       TargetEncoder(**_tgt_kw),  HIGH_CARD),
    ("num",       "passthrough",             NUMERICAL_FEATURES),
    ("tfidf_icg", TfidfVectorizer(**tfidf_kw(_txt["tfidf_icg"])), TEXT_COL),
    ("tfidf_ice", TfidfVectorizer(**tfidf_kw(_txt["tfidf_ice"])), TEXT_COL_2),
    ("tfidf_df",  TfidfVectorizer(**tfidf_kw(_txt["tfidf_df"])),  TEXT_COL_3),
], remainder="drop")

# (1) Structural fit — sets n_features_in_, feature_names_in_, sparse_output_,
#     _remainder and all other sklearn fitted-state metadata.
preprocessor_t2.fit(X_tr_t2.iloc[:_SAMPLE], y_tr[:_SAMPLE])
gc.collect()

# (2) Fit sub-transformers on full data one at a time to cap peak memory.
_ord_enc   = OrdinalEncoder(**_ord_kw).fit(X_tr_t2[LOW_CARD])
gc.collect()
_tgt_enc   = TargetEncoder(**_tgt_kw).fit(X_tr_t2[HIGH_CARD], y_tr)
gc.collect()
_tfidf_icg = TfidfVectorizer(**tfidf_kw(_txt["tfidf_icg"])).fit(X_tr_t2[TEXT_COL])
gc.collect()
_tfidf_ice = TfidfVectorizer(**tfidf_kw(_txt["tfidf_ice"])).fit(X_tr_t2[TEXT_COL_2])
gc.collect()
_tfidf_df  = TfidfVectorizer(**tfidf_kw(_txt["tfidf_df"])).fit(X_tr_t2[TEXT_COL_3])
gc.collect()

# (3) Inject properly-fitted estimators.
#     Mutate only the estimator slot of each entry; preserve the column spec and
#     the passthrough entry (index 2) exactly as sklearn stored them after the
#     structural fit — avoids "str has no attribute transform" from a raw "passthrough".
_t2_ = list(preprocessor_t2.transformers_)
for i, (name, _, cols) in enumerate(_t2_):
    if name == "ord":
        _t2_[i] = (name, _ord_enc, cols)
    elif name == "tgt":
        _t2_[i] = (name, _tgt_enc, cols)
    elif name == "tfidf_icg":
        _t2_[i] = (name, _tfidf_icg, cols)
    elif name == "tfidf_ice":
        _t2_[i] = (name, _tfidf_ice, cols)
    elif name == "tfidf_df":
        _t2_[i] = (name, _tfidf_df, cols)
    # "num" (passthrough) and any unknown names are left untouched
preprocessor_t2.transformers_ = _t2_
del _t2_
print("Tier-2 preprocessor fitted.")

# (4) Probe shape on 10 rows — avoids materialising the full ~2.8 GB sparse matrix.
_probe = preprocessor_t2.transform(X_tr_t2.iloc[:10])
print(f"  T2 train matrix: ({len(X_tr_t2)}, {_probe.shape[1]})  sparse={sp.issparse(_probe)}")
print(f"  ({_txt['tfidf_icg']['max_features']} ICG + {_txt['tfidf_ice']['max_features']} ICE + {_txt['tfidf_df']['max_features']} DF TF-IDF dims)")
del _probe; gc.collect()


## TP-3 · Shared Helpers

Define the global `results` dictionary that accumulates test-set metrics for every
model we train, and instantiate a `StratifiedKFold` splitter for all cross-validation
runs. The `evaluate()` helper (imported from `category_classification.py`) populates
`results` with macro F1, balanced accuracy, and optional CV statistics.

### TP-3.1 · Evaluation Helper

Define `evaluate`, a shared function that computes macro F1 and balanced accuracy,
stores results in the global `results` dict, and prints a one-line summary. Every
model section (Dummy, Logistic Regression, LightGBM flat, Two-Stage) calls this
function so metrics are collected in a uniform schema for the final comparison table.

In [ ]:
results = {}  # {name: {macro_f1, bal_acc, tier, approach[, cv_f1_mean, cv_f1_std]}}
# Stratified folds keep class proportions identical across splits — essential for
# imbalanced categories where a random split could orphan minority classes.
cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=False)
# evaluate() imported from category_classification — pass results=results at each call site
print("Evaluation helper ready.")

## TP-4 · Tier 1 — Structured Features Only

Train and evaluate models that use **only** the structured engineered features
(ordinal/target-encoded categoricals + numericals). This tier establishes a
performance floor and ceiling before adding expensive text features.

### TP-4.1 · Baselines — Dummy & Logistic Regression

Fit two fast baselines on Tier-1 data:

1. **DummyClassifier** (`most_frequent`) — predicts the majority class every time.
   Gives the absolute floor for macro F1.
2. **LogisticRegression** — a linear model with L2 regularisation and class weights.
   Serves as a strong, interpretable baseline before tree-based boosting.

In [ ]:
# ── Dummy baseline ────────────────────────────────────────────────────────────
_dummy = cfg["models"]["dummy"]
dummy = DummyClassifier(strategy=_dummy["strategy"], random_state=RANDOM_STATE)
dummy.fit(X_tr, y_tr)
# Cross-validation on training data to gauge variance (Dummy has none, but kept for consistency)
_dummy_cv = cross_val_score(dummy, X_tr, y_tr, cv=cv, scoring="f1_macro", n_jobs=-1)
evaluate("Dummy (most_frequent)", y_te, dummy.predict(X_te),
         tier="0", approach="baseline", results=results, cv_scores=_dummy_cv)

# ── Logistic Regression (flat, Tier 1) ────────────────────────────────────────
_lr = cfg["models"]["logistic_regression"]
lr_pipe = Pipeline([
    ("pre", preprocessor_t1),   # ordinal / target / passthrough
    ("scl", StandardScaler()),  # centre & scale before linear solver
    ("clf", LogisticRegression(
        solver=_lr.get("solver", "saga"),
        max_iter=_lr["max_iter"],
        tol=_lr.get("tol", 1e-3),
        class_weight=_lr["class_weight"],  # up-weight rare categories
        random_state=RANDOM_STATE,
        n_jobs=_lr["n_jobs"],
    )),
])
lr_pipe.fit(X_tr, y_tr)
# Use fewer CV folds for LR because it is slower on high-dimensional encoded data
_lr_cv = cross_val_score(
    lr_pipe, X_tr, y_tr,
    cv=StratifiedKFold(cfg["training"].get("lr_cv_folds", 3), shuffle=False),
    scoring="f1_macro", n_jobs=-1,
)
evaluate("LogisticRegression (T1, flat)", y_te, lr_pipe.predict(X_te),
         tier="1", approach="flat", results=results, cv_scores=_lr_cv)


### TP-4.2 · LightGBM (Tier 1, flat search)

Run `HalvingRandomSearchCV` over a broad hyper-parameter space for LightGBM using
the Tier-1 preprocessor. Halving search is efficient: it evaluates many random
candidates on a small subset of data, promotes the best ones, and retrains on
progressively larger subsets, yielding a strong configuration in far less wall-time
than exhaustive grid search.

In [ ]:
# Build the search-space dict from TOML; make_dist turns [low, high] into scipy distributions
_lgb = cfg["models"]["lightgbm"]
lgbm_t1 = Pipeline([
    ("pre", preprocessor_t1),
    ("clf", LGBMClassifier(objective=_lgb["objective"], num_class=n_classes,
                           class_weight=_lgb["class_weight"], random_state=RANDOM_STATE,
                           n_jobs=_lgb["n_jobs"], verbose=_lgb["verbose"],
                           device=_LGB_DEVICE)),
])
param_lgbm = {
    f"clf__{k}": make_dist(v)
    for k, v in _lgb["search_space"].items()
}
# HalvingRandomSearchCV settings: n_candidates initial draws, factor=reduction per iteration
_hs = cfg["training"]["halving_search"]
search_lgbm_t1 = HalvingRandomSearchCV(
    lgbm_t1, param_lgbm, n_candidates=_hs["n_candidates"], factor=_hs["factor"],
    cv=cv, scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=-1, verbose=0,
    return_train_score=True,  # lets us diagnose overfit (train vs CV gap)
)
search_lgbm_t1.fit(X_tr, y_tr)
print(f"LightGBM T1 best CV macro F1: {search_lgbm_t1.best_score_:.4f}")
print(f"Best params: {search_lgbm_t1.best_params_}")
evaluate("LightGBM (T1, flat)", y_te, search_lgbm_t1.predict(X_te), tier="1", approach="flat", results=results)


### TP-4.3 · Two-Stage LightGBM (Tier 1)

Train a **Two-Stage** classifier: Stage 1 is a binary model that decides
"OTHER vs NOT-OTHER"; Stage 2 is a multi-class model trained only on the
non-OTHER subset. This architecture can improve recall on rare categories
because the second-stage classifier does not have to compete with the
dominant OTHER class for gradient weight.

In [ ]:
# Load two-stage hyper-parameters (lighter than the flat-search config)
_ts = cfg["models"]["lightgbm"]["two_stage"]
ts_clf_t1 = TwoStageClassifier(
    stage1=LGBMClassifier(n_estimators=_ts["n_estimators"], learning_rate=_ts["learning_rate"],
                          num_leaves=_ts["num_leaves"], class_weight=_ts["class_weight"],
                          random_state=RANDOM_STATE, n_jobs=_ts["n_jobs"], verbose=_ts["verbose"],
                          device=_LGB_DEVICE),
    stage2=LGBMClassifier(n_estimators=_ts["n_estimators"], learning_rate=_ts["learning_rate"],
                          num_leaves=_ts["num_leaves"], class_weight=_ts["class_weight"],
                          random_state=RANDOM_STATE, n_jobs=_ts["n_jobs"], verbose=_ts["verbose"],
                          device=_LGB_DEVICE),
    other_label=other_idx,  # The label index that represents the catch-all OTHER class
)
ts_pipe_t1 = Pipeline([("pre", preprocessor_t1), ("clf", ts_clf_t1)])
ts_pipe_t1.fit(X_tr, y_tr)
evaluate("LightGBM (T1, two-stage)", y_te, ts_pipe_t1.predict(X_te),
         tier="1", approach="two-stage", results=results)

# Compute Stage-1 binary F1 separately to diagnose the quality of the OTHER filter
X_te_pre_t1 = preprocessor_t1.transform(X_te)
y_te_bin = (y_te != other_idx).astype(int)
s1_f1 = f1_score(y_te_bin, ts_clf_t1.stage1.predict(X_te_pre_t1), average="binary")
print(f"  Stage-1 binary F1 (NOT-OTHER detection): {s1_f1:.4f}")


## TP-5 · Tier 2 — Structured Features + TF-IDF on All Three Text Columns

Repeat the LightGBM flat search and two-stage training, but this time using the
Tier-2 preprocessor which concatenates structured features with TF-IDF vectors
from `incompatible_content_ground`, `incompatible_content_explanation`, and
`decision_facts`. The dimensionality jumps from ~50 to ~6 000+; we expect a
large F1 boost for text-rich categories.

In [ ]:
# Re-define param_lgbm locally so this cell can run independently of TP-4.2
param_lgbm = {
    f"clf__{k}": make_dist(v)
    for k, v in cfg["models"]["lightgbm"]["search_space"].items()
}
# Merge fixed LightGBM args from TOML, overriding search-space defaults where needed
_lgb    = cfg["models"]["lightgbm"]
_lgb_ff = cfg["models"]["lightgbm"]["tier_flat_fixed"]
_lgb_fix = {k: v for k, v in _lgb.items() if k not in ("search_space", "two_stage", "tier_flat_fixed")}
_lgb_t2  = {**_lgb_fix,
             "n_estimators": _lgb_ff["n_estimators"],
             "learning_rate": _lgb_ff["learning_rate"],
             "num_leaves": _lgb_ff["num_leaves"],
             "num_class": n_classes, "device": _LGB_DEVICE}
lgbm_t2_flat = Pipeline([
    ("pre", preprocessor_t2),   # structured + TF-IDF on 3 text columns
    ("clf", LGBMClassifier(**_lgb_t2)),
])
_hs = cfg["training"]["halving_search"]
search_lgbm_t2 = HalvingRandomSearchCV(
    lgbm_t2_flat, param_lgbm, n_candidates=_hs["n_candidates"], factor=_hs["factor"],
     cv=StratifiedKFold(N_CV_FOLDS, shuffle=False),
    scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=-1, verbose=0,
)
search_lgbm_t2.fit(X_tr_t2, y_tr)
print(f"LightGBM T2 best CV macro F1: {search_lgbm_t2.best_score_:.4f}")
evaluate("LightGBM (T2, flat)", y_te, search_lgbm_t2.predict(X_te_t2), tier="2", approach="flat", results=results)


### TP-5.1 · Two-Stage LightGBM (Tier 2)

Train a two-stage classifier using the Tier-2 preprocessor (structured + TF-IDF on
all three text columns). The Stage-1 binary filter is applied to the full T2 feature
matrix, and Stage-2 is trained on the non-OTHER subset. We compare Tier-1 vs. Tier-2
macro F1 side-by-side in the next section (TP-6).

In [ ]:
# Load two-stage hyper-parameters from TOML (same architecture as Tier 1)
_ts = cfg["models"]["lightgbm"]["two_stage"]
ts_clf_t2 = TwoStageClassifier(
    stage1=LGBMClassifier(n_estimators=_ts["n_estimators"], learning_rate=_ts["learning_rate"],
                          num_leaves=_ts["num_leaves"], class_weight=_ts["class_weight"],
                          random_state=RANDOM_STATE, n_jobs=_ts["n_jobs"], verbose=_ts["verbose"],
                          device=_LGB_DEVICE),
    stage2=LGBMClassifier(n_estimators=_ts["n_estimators"], learning_rate=_ts["learning_rate"],
                          num_leaves=_ts["num_leaves"], class_weight=_ts["class_weight"],
                          random_state=RANDOM_STATE, n_jobs=_ts["n_jobs"], verbose=_ts["verbose"],
                          device=_LGB_DEVICE),
    other_label=other_idx,
)
ts_pipe_t2 = Pipeline([("pre", preprocessor_t2), ("clf", ts_clf_t2)])
ts_pipe_t2.fit(X_tr_t2, y_tr)
evaluate("LightGBM (T2, two-stage)", y_te, ts_pipe_t2.predict(X_te_t2),
         tier="2", approach="two-stage", results=results)

## TP-6 · Model Comparison

Aggregate the `results` dict into a sorted DataFrame and visualise every model's
macro F1 and balanced accuracy. The bar charts are colour-coded by tier so it is
immediately obvious how much Tier-2 text features improve over Tier-1 structured-only
and over the Dummy baseline.

In [ ]:
# Assemble results into a DataFrame for sorting and plotting
df_all = pd.DataFrame(results).T.reset_index().rename(columns={"index": "model"})
df_all["macro_f1"] = df_all["macro_f1"].astype(float)
df_all["bal_acc"]  = df_all["bal_acc"].astype(float)
df_all = df_all.sort_values("macro_f1", ascending=False)

print("=== All models — macro F1 (descending) ===")
print(df_all[["model","tier","approach","macro_f1","bal_acc"]].to_string(index=False))

# Colour map: tier 0 = baseline grey, 1 = blue, 2 = green, 3 = amber
tier_colors = {"0": "#8892b0", "1": "#6c8fff", "2": "#34d399", "3": "#fbbf24"}
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, metric, title in zip(axes, ["macro_f1","bal_acc"],
                              ["Macro F1 (primary)", "Balanced Accuracy"]):
    df_plot = df_all.sort_values(metric, ascending=True)
    bars = ax.barh(df_plot["model"], df_plot[metric],
                   color=[tier_colors.get(str(t),"#ccc") for t in df_plot["tier"]])
    ax.set_xlabel(metric); ax.set_title(title)
    # Red dashed line = Dummy baseline so the gap is visually obvious
    ax.axvline(float(results["Dummy (most_frequent)"][metric]),
               ls="--", color="red", alpha=0.6, label="Dummy")
    ax.legend()
    for bar, v in zip(bars, df_plot[metric]):
        ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
                f"{v:.4f}", va="center", fontsize=8)
axes[0].legend(handles=[Patch(color=c, label=f"Tier {t}") for t,c in tier_colors.items()],
               loc="lower right")
plt.suptitle("All models — full comparison", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


## TP-7 · Best Model — Detailed Evaluation

Identify the model with the highest macro F1 from `results`, retrieve its fitted
pipeline, generate predictions on the held-out test set, and print a full
classification report. This section is diagnostic: it tells us *which* categories
are hard and *how* the best model confuses them.

In [ ]:
# Pick the best model by macro F1 (DataFrame already sorted descending in TP-6)
best_name = df_all.iloc[0]["model"]
best_tier = df_all.iloc[0]["tier"]
best_macro_f1 = float(df_all.iloc[0]["macro_f1"])
best_bal_acc  = float(df_all.iloc[0]["bal_acc"])
print(f"Best model: {best_name}  (Tier {best_tier})")

# Map display names back to the in-memory pipeline objects trained above
model_map = {
    "Dummy (most_frequent)":            dummy,
    "LogisticRegression (T1, flat)":    lr_pipe,
    "LightGBM (T1, flat)":              search_lgbm_t1.best_estimator_,
    "LightGBM (T1, two-stage)":         ts_pipe_t1,
    "LightGBM (T2, flat)":              search_lgbm_t2.best_estimator_,
    "LightGBM (T2, two-stage)":         ts_pipe_t2,
}

best_pipe = model_map.get(best_name)
if best_pipe is None:
    raise KeyError(f"best_name '{best_name}' not found in model_map — add it explicitly.")

# predict_dispatch imported from category_classification handles tier switching
y_best_pred, y_best_proba = predict_dispatch(
    best_pipe, best_name, best_tier, X_te,
    X_t2=X_te_t2)
print()
print(classification_report(y_te, y_best_pred, target_names=target_labels, zero_division=0))

### TP-7.1 · Confusion Matrix

Plot the raw-count and row-normalised confusion matrices side-by-side.
Row normalisation shows what fraction of each true class is sent to each predicted
class, independent of class size. This is especially useful for imbalanced data
where a raw-count matrix would be dominated by the majority class.

In [ ]:
# Confusion matrix — normalised and raw
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
short = [l[:18] for l in target_labels]  # truncated labels for readability
for ax, norm in zip(axes, [None, "true"]):
    cm_arr = confusion_matrix(y_te, y_best_pred,
                              normalize=norm if norm else None)
    sns.heatmap(cm_arr, annot=True, fmt=("d" if norm is None else ".2f"),
                cmap="Blues", ax=ax, xticklabels=short, yticklabels=short)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix — {'counts' if norm is None else 'row-normalised'}")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.suptitle(best_name, fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

### TP-7.2 · Per-Class F1 Score

Visualise F1 for every category, colour-coded by performance tier:
red < 0.3 (poor), amber < 0.6 (moderate), green ≥ 0.6 (good).
Classes in red are the primary targets for further improvement (more samples,
better text features, or class-specific thresholds).

In [ ]:
# Per-class F1 — sorted ascending so the worst performers appear at the top
per_f1 = f1_score(y_te, y_best_pred, average=None, zero_division=0)
f1_s   = pd.Series(per_f1, index=target_labels).sort_values()
fig, ax = plt.subplots(figsize=(12, 5))
colors_f1 = ["#f87171" if v < 0.3 else "#fbbf24" if v < 0.6 else "#34d399" for v in f1_s.values]
ax.barh(f1_s.index, f1_s.values, color=colors_f1)
ax.axvline(f1_s.mean(), ls="--", color="grey", label=f"Mean {f1_s.mean():.3f}")
ax.set_xlabel("F1 score"); ax.set_title(f"Per-class F1 — {best_name}"); ax.legend()
for i, v in enumerate(f1_s.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()


### TP-7.3 · Top Misclassification Pairs

List the 20 most frequent true→predicted misclassification pairs together with
their count and percentage of the true class's test rows. These pairs reveal which
category boundaries are most ambiguous and should be prioritised for manual label
review or feature-engineering investment.

In [ ]:
# Top misclassification pairs
cm_mc = confusion_matrix(y_te, y_best_pred)
pairs = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm_mc[i, j] > 0:
            pairs.append({"True": target_labels[i], "Predicted": target_labels[j],
                          "Count": int(cm_mc[i, j]),
                          "% of true": f"{100*cm_mc[i,j]/cm_mc[i].sum():.1f}%"})
err_df = pd.DataFrame(pairs).sort_values("Count", ascending=False).head(20)
print("Top 20 misclassification pairs:")
display(err_df)


## TP-8 · Save Model Artifact & Upload to Hopsworks Model Registry

Serialise the best model pipeline, both preprocessors, label maps, column lists,
and platform automation rates into a single pickle artifact (`category_best_model.pkl`).
Then copy the artifact to the export directory and register it in the Hopsworks Model
Registry so the Inference Pipeline can discover and download it by name/version.

In [ ]:
save_obj = {
    "name":          best_name,
    "tier":          best_tier,
    "macro_f1":      best_macro_f1,
    "target_labels": target_labels,
    "label_map":     label_map,
    "label_map_inv": label_map_inv,
    "other_label":   other_idx,
    "FEATURE_COLS":  FEATURE_COLS,
    "CATEGORICAL_FEATURES": CATEGORICAL_FEATURES,
    "NUMERICAL_FEATURES":   NUMERICAL_FEATURES,
    "TEXT_COL":      TEXT_COL,
    "TEXT_COL_2":    TEXT_COL_2,
    "TEXT_COL_3":    TEXT_COL_3,
    "preprocessor_t1": preprocessor_t1,
    "preprocessor_t2": preprocessor_t2,
    # Training-era per-platform automation rate so inference
    # applies the same values the model was trained on.
    "platform_automation_rates": PLATFORM_AUTOMATION_RATES,
}

if best_pipe is None:
    raise KeyError(f"best_name '{best_name}' not found in model_map — add it explicitly.")
save_obj["pipeline"] = best_pipe

# Write the artifact locally
with open(MODEL_ARTIFACT, "wb") as f:
    pickle.dump(save_obj, f)
print(f"Model artifact saved → {MODEL_ARTIFACT}  "
      f"({MODEL_ARTIFACT.stat().st_size / 1e6:.1f} MB)")

# Stage a copy for Hopsworks upload (registry expects a directory)
MODEL_EXPORT_DIR.mkdir(exist_ok=True)
shutil.copy(MODEL_ARTIFACT, MODEL_EXPORT_DIR / MODEL_ARTIFACT.name)

# Re-connect to Hopsworks and push to the Model Registry
project = hopsworks.login(
    host=HOPSWORKS_HOST,
    project=HOPSWORKS_PROJECT,
    api_key_value=HOPSWORKS_API_KEY,
)
mr = project.get_model_registry()

hops_model = mr.python.create_model(
    name=MODEL_NAME,
    metrics={"macro_f1": best_macro_f1, "balanced_accuracy": best_bal_acc},
    description=(
        f"Category classifier: {best_name}  |  Tier {best_tier}  "
        f"|  {n_classes} classes  |  macro F1={best_macro_f1:.4f}"
    ),
)
hops_model.save(str(MODEL_EXPORT_DIR))
print(f"Model uploaded to Hopsworks Model Registry: '{MODEL_NAME}'  "
      f"version={hops_model.version}")

## TP-9 · Summary

Auto-generate a markdown report summarising the training run: timestamp,
train/test split sizes, best model name, tier, macro F1, balanced accuracy,
and a table of all evaluated models. This cell can be copied directly into
experiment-tracking tickets or documentation.

In [ ]:

lines = [
    "## Task A — Category Classification Results",
    "",
    f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M')}  ",
    f"**Train/Test:** {len(X_tr):,} / {len(X_te):,}  "
    f"|  **Primary metric:** macro F1  ",
    "",
    "### Best model",
    "| | |", "|---|---|",
    f"| Model | {best_name} |",
    f"| Tier  | {best_tier} |",
    f"| Macro F1 | {best_macro_f1:.4f} |",
    f"| Balanced accuracy | {best_bal_acc:.4f} |",
    f"| Dummy baseline macro F1 | {results['Dummy (most_frequent)']['macro_f1']:.4f} |",
    "",
    "### All results",
    "| Model | Tier | Approach | Macro F1 | Balanced Acc |",
    "|---|---|---|---|---|",
]
# Append one markdown table row per model, sorted by macro F1 descending
for _, row in df_all.iterrows():
    lines.append(f"| {row['model']} | {row['tier']} | {row['approach']} "
                 f"| {float(row['macro_f1']):.4f} | {float(row['bal_acc']):.4f} |")

display(Markdown("\n".join(lines)))
